# EXP-006: 상위 노트북 파이프라인 재현

**현재 LB 최고:** 0.96597 (XGB 단독) → **목표:** 0.97+

핵심 변경점:
1. 171개 전체 페어와이즈 조합 (수치형+범주형) → 카디널리티 필터 → ~135개
2. Fold 내 TargetEncoder (train_fold ∪ 원본데이터로 fit)
3. 3모델 앙상블 + Bias Tuning

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import lightgbm as lgb_module
from itertools import combinations
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder, TargetEncoder
from sklearn.utils.class_weight import compute_sample_weight
from scipy.special import logit, expit
import warnings
warnings.filterwarnings('ignore')

mpl.rcParams['font.family'] = 'Malgun Gothic'
mpl.rcParams['axes.unicode_minus'] = False

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
orig = pd.read_csv('../data/irrigation_prediction.csv')

TARGET = 'Irrigation_Need'
CAT_COLS = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season',
            'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
NUM_COLS = ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity',
            'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours',
            'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']
ALL_SOURCE = NUM_COLS + CAT_COLS

print(f"Train: {train.shape}, Test: {test.shape}, Orig: {orig.shape}")

Train: (630000, 21), Test: (270000, 20), Orig: (10000, 20)


## 1. 페어와이즈 조합 생성 (171개 → 필터 → ~135개)

In [2]:
# 타겟 인코딩
target_le = LabelEncoder()
target_le.fit(train[TARGET])
y_train = target_le.transform(train[TARGET])
y_orig = target_le.transform(orig[TARGET])

# 전체 페어와이즈 조합 생성 + 카디널리티 필터
def build_pair_columns(tr, te, og):
    """171개 쌍 생성 → 카디널리티 필터 → 유지된 쌍 반환"""
    pair_cols = []
    total = len(tr) + len(te) + len(og)
    thresh = total // 2

    for left, right in combinations(ALL_SOURCE, 2):
        name = f"{left}__{right}"
        tr[name] = tr[left].astype(str) + '_' + tr[right].astype(str)
        te[name] = te[left].astype(str) + '_' + te[right].astype(str)
        og[name] = og[left].astype(str) + '_' + og[right].astype(str)

        # 공동 factorize
        combined = pd.concat([tr[name], te[name], og[name]], ignore_index=True)
        codes, _ = pd.factorize(combined)
        n_unique = pd.Series(codes).nunique()

        if n_unique > thresh:
            tr.drop(columns=[name], inplace=True)
            te.drop(columns=[name], inplace=True)
            og.drop(columns=[name], inplace=True)
            continue

        # factorize 적용
        tr[name] = codes[:len(tr)].astype('int32')
        te[name] = codes[len(tr):len(tr)+len(te)].astype('int32')
        og[name] = codes[len(tr)+len(te):].astype('int32')
        pair_cols.append(name)

    return pair_cols

pair_cols = build_pair_columns(train, test, orig)
print(f"생성된 쌍: C(19,2)=171 → 필터 후 유지: {len(pair_cols)}개")

생성된 쌍: C(19,2)=171 → 필터 후 유지: 135개


In [3]:
# 기본 피처 준비
# 원본 범주형은 LabelEncoding
for col in CAT_COLS:
    le = LabelEncoder()
    all_vals = pd.concat([train[col], test[col], orig[col]])
    le.fit(all_vals)
    train[col] = le.transform(train[col])
    test[col] = le.transform(test[col])
    orig[col] = le.transform(orig[col])

# TargetEncoder 대상 컬럼 = pair_cols (factorized)
te_cols = pair_cols

# 모델 학습에 사용할 base 피처 (수치형 + LabelEncoded 범주형 + factorized 쌍)
drop_cols = ['id', TARGET]
base_cols = [c for c in train.columns if c not in drop_cols]

# 원본 데이터 설정
ORIG_WEIGHT = 0.35
N_FOLDS = 10
SEED = 42

print(f"Base 피처: {len(base_cols)}개")
print(f"TargetEncoder 대상: {len(te_cols)}개 (페어와이즈 조합)")
print(f"TE 적용 후 총 피처: {len(base_cols) + len(te_cols)*3}개 (각 조합 × 3클래스)")

Base 피처: 154개
TargetEncoder 대상: 135개 (페어와이즈 조합)
TE 적용 후 총 피처: 559개 (각 조합 × 3클래스)


## 2. Step 1: XGBoost (Fold 내 TargetEncoder + 원본 데이터)

In [4]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

X_train_base = train[base_cols]
X_test_base = test[base_cols]
X_orig_base = orig[base_cols]

xgb_params = {
    'n_estimators': 5000,
    'max_depth': 6,
    'learning_rate': 0.02,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 5,
    'gamma': 0.1,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'objective': 'multi:softprob',
    'num_class': 3,
    'eval_metric': 'mlogloss',
    'tree_method': 'hist',
    'device': 'cuda',
    'random_state': SEED,
    'n_jobs': -1,
    'verbosity': 0,
    'early_stopping_rounds': 200,
}

oof_xgb = np.zeros((len(train), 3))
test_xgb = np.zeros((len(test), 3))

print("=== XGBoost + 전체 파이프라인 ===")
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train_base, y_train)):
    # 1) train_fold ∪ orig 결합
    X_tr_base = pd.concat([X_train_base.iloc[tr_idx], X_orig_base], ignore_index=True)
    y_tr = np.concatenate([y_train[tr_idx], y_orig])

    X_va_base = X_train_base.iloc[va_idx]
    y_va = y_train[va_idx]

    # 2) TargetEncoder: 결합셋으로 fit
    encoder = TargetEncoder(target_type='multiclass', cv=5, random_state=SEED, smooth='auto')
    te_tr = encoder.fit_transform(X_tr_base[te_cols], y_tr)
    te_va = encoder.transform(X_va_base[te_cols])
    te_te = encoder.transform(X_test_base[te_cols])

    # TE 피처 이름
    te_names = [f"te_{col}_c{c}" for col in te_cols for c in range(3)]

    # 3) base + TE 결합
    X_tr_full = pd.concat([X_tr_base.reset_index(drop=True),
                           pd.DataFrame(te_tr, columns=te_names)], axis=1)
    X_va_full = pd.concat([X_va_base.reset_index(drop=True),
                           pd.DataFrame(te_va, columns=te_names)], axis=1)
    X_te_full = pd.concat([X_test_base.reset_index(drop=True),
                           pd.DataFrame(te_te, columns=te_names)], axis=1)

    # 4) sample_weight: balanced + orig 다운웨이트
    sw = compute_sample_weight('balanced', y_tr)
    n_train_fold = len(tr_idx)
    sw[n_train_fold:] *= ORIG_WEIGHT

    # 5) 학습
    model = XGBClassifier(**xgb_params)
    model.fit(X_tr_full, y_tr, sample_weight=sw,
              eval_set=[(X_va_full, y_va)], verbose=0)

    oof_xgb[va_idx] = model.predict_proba(X_va_full)
    test_xgb += model.predict_proba(X_te_full) / N_FOLDS

    score = balanced_accuracy_score(y_va, oof_xgb[va_idx].argmax(axis=1))
    print(f"  Fold {fold}: {score:.5f} (best_iter: {model.best_iteration})")

xgb_score = balanced_accuracy_score(y_train, oof_xgb.argmax(axis=1))
print(f"\n  XGBoost OOF: {xgb_score:.5f} (이전 최고: 0.96812)")

=== XGBoost + 전체 파이프라인 ===
  Fold 0: 0.97423 (best_iter: 2614)
  Fold 1: 0.97483 (best_iter: 2541)
  Fold 2: 0.97410 (best_iter: 2313)
  Fold 3: 0.97647 (best_iter: 2554)
  Fold 4: 0.97653 (best_iter: 2728)
  Fold 5: 0.97665 (best_iter: 2369)
  Fold 6: 0.97625 (best_iter: 2440)
  Fold 7: 0.97532 (best_iter: 2654)
  Fold 8: 0.97418 (best_iter: 2293)
  Fold 9: 0.97568 (best_iter: 2538)

  XGBoost OOF: 0.97542 (이전 최고: 0.96812)


## 3. Step 2: LightGBM + CatBoost (동일 파이프라인)

In [5]:
# --- LightGBM ---
oof_lgb = np.zeros((len(train), 3))
test_lgb = np.zeros((len(test), 3))

print("=== LightGBM + 전체 파이프라인 ===")
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train_base, y_train)):
    X_tr_base = pd.concat([X_train_base.iloc[tr_idx], X_orig_base], ignore_index=True)
    y_tr = np.concatenate([y_train[tr_idx], y_orig])
    X_va_base = X_train_base.iloc[va_idx]
    y_va = y_train[va_idx]

    encoder = TargetEncoder(target_type='multiclass', cv=5, random_state=SEED, smooth='auto')
    te_tr = encoder.fit_transform(X_tr_base[te_cols], y_tr)
    te_va = encoder.transform(X_va_base[te_cols])
    te_te = encoder.transform(X_test_base[te_cols])

    te_names = [f"te_{col}_c{c}" for col in te_cols for c in range(3)]
    X_tr_full = pd.concat([X_tr_base.reset_index(drop=True),
                           pd.DataFrame(te_tr, columns=te_names)], axis=1)
    X_va_full = pd.concat([X_va_base.reset_index(drop=True),
                           pd.DataFrame(te_va, columns=te_names)], axis=1)
    X_te_full = pd.concat([X_test_base.reset_index(drop=True),
                           pd.DataFrame(te_te, columns=te_names)], axis=1)

    sw = compute_sample_weight('balanced', y_tr)
    sw[len(tr_idx):] *= ORIG_WEIGHT

    model = LGBMClassifier(
        n_estimators=3000,
        max_depth=-1,
        num_leaves=127,
        learning_rate=0.04,
        subsample=0.8,
        colsample_bytree=0.6,
        reg_alpha=0.1,
        reg_lambda=1.0,
        min_child_samples=20,
        objective='multiclass',
        num_class=3,
        metric='multi_logloss',
        random_state=SEED,
        n_jobs=-1,
        verbosity=-1,
    )
    model.fit(X_tr_full, y_tr, sample_weight=sw,
              eval_set=[(X_va_full, y_va)],
              callbacks=[lgb_module.early_stopping(150, verbose=False)])

    oof_lgb[va_idx] = model.predict_proba(X_va_full)
    test_lgb += model.predict_proba(X_te_full) / N_FOLDS

    score = balanced_accuracy_score(y_va, oof_lgb[va_idx].argmax(axis=1))
    print(f"  Fold {fold}: {score:.5f} (best_iter: {model.best_iteration_})")

lgb_score = balanced_accuracy_score(y_train, oof_lgb.argmax(axis=1))
print(f"\n  LightGBM OOF: {lgb_score:.5f} (이전 최고: 0.96572)")

=== LightGBM + 전체 파이프라인 ===
  Fold 0: 0.97362 (best_iter: 349)
  Fold 1: 0.97473 (best_iter: 343)
  Fold 2: 0.97411 (best_iter: 338)
  Fold 3: 0.97628 (best_iter: 341)
  Fold 4: 0.97632 (best_iter: 347)
  Fold 5: 0.97598 (best_iter: 340)
  Fold 6: 0.97691 (best_iter: 340)
  Fold 7: 0.97403 (best_iter: 354)
  Fold 8: 0.97413 (best_iter: 339)
  Fold 9: 0.97532 (best_iter: 346)

  LightGBM OOF: 0.97514 (이전 최고: 0.96572)


In [6]:
# --- CatBoost ---
oof_cat = np.zeros((len(train), 3))
test_cat = np.zeros((len(test), 3))

print("=== CatBoost + 전체 파이프라인 ===")
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train_base, y_train)):
    X_tr_base = pd.concat([X_train_base.iloc[tr_idx], X_orig_base], ignore_index=True)
    y_tr = np.concatenate([y_train[tr_idx], y_orig])
    X_va_base = X_train_base.iloc[va_idx]
    y_va = y_train[va_idx]

    encoder = TargetEncoder(target_type='multiclass', cv=5, random_state=SEED, smooth='auto')
    te_tr = encoder.fit_transform(X_tr_base[te_cols], y_tr)
    te_va = encoder.transform(X_va_base[te_cols])
    te_te = encoder.transform(X_test_base[te_cols])

    te_names = [f"te_{col}_c{c}" for col in te_cols for c in range(3)]
    X_tr_full = pd.concat([X_tr_base.reset_index(drop=True),
                           pd.DataFrame(te_tr, columns=te_names)], axis=1)
    X_va_full = pd.concat([X_va_base.reset_index(drop=True),
                           pd.DataFrame(te_va, columns=te_names)], axis=1)
    X_te_full = pd.concat([X_test_base.reset_index(drop=True),
                           pd.DataFrame(te_te, columns=te_names)], axis=1)

    sw = compute_sample_weight('balanced', y_tr)
    sw[len(tr_idx):] *= ORIG_WEIGHT

    model = CatBoostClassifier(
        iterations=2000,
        depth=8,
        learning_rate=0.04,
        l2_leaf_reg=4.0,
        loss_function='MultiClass',
        eval_metric='MultiClass',
        task_type='GPU',
        random_seed=SEED,
        verbose=0,
        early_stopping_rounds=150,
    )
    model.fit(X_tr_full, y_tr, sample_weight=sw,
              eval_set=(X_va_full, y_va))

    oof_cat[va_idx] = model.predict_proba(X_va_full)
    test_cat += model.predict_proba(X_te_full) / N_FOLDS

    score = balanced_accuracy_score(y_va, oof_cat[va_idx].argmax(axis=1))
    print(f"  Fold {fold}: {score:.5f} (best_iter: {model.get_best_iteration()})")

cat_score = balanced_accuracy_score(y_train, oof_cat.argmax(axis=1))
print(f"\n  CatBoost OOF: {cat_score:.5f} (이전 최고: 0.96820)")

=== CatBoost + 전체 파이프라인 ===
  Fold 0: 0.97702 (best_iter: 1999)
  Fold 1: 0.97536 (best_iter: 1999)
  Fold 2: 0.97696 (best_iter: 1999)
  Fold 3: 0.97774 (best_iter: 1999)
  Fold 4: 0.97926 (best_iter: 1999)
  Fold 5: 0.97838 (best_iter: 1997)
  Fold 6: 0.97785 (best_iter: 1999)
  Fold 7: 0.97657 (best_iter: 1999)
  Fold 8: 0.97612 (best_iter: 1999)
  Fold 9: 0.97850 (best_iter: 1999)

  CatBoost OOF: 0.97738 (이전 최고: 0.96820)


## 4. Step 3: 블렌딩 + Bias Tuning

In [7]:
print("=" * 55)
print(f"{'모델':<15} {'OOF':>10}")
print("=" * 55)
print(f"{'XGBoost':<15} {xgb_score:>10.5f}")
print(f"{'LightGBM':<15} {lgb_score:>10.5f}")
print(f"{'CatBoost':<15} {cat_score:>10.5f}")
print("=" * 55)

# 균등 블렌딩
blend_equal = (oof_xgb + oof_lgb + oof_cat) / 3
score_equal = balanced_accuracy_score(y_train, blend_equal.argmax(axis=1))

# 가중치 그리드 서치
best_w_score = 0
best_w = None
for w1 in np.arange(0.10, 0.80, 0.05):
    for w2 in np.arange(0.05, 0.80 - w1, 0.05):
        w3 = 1.0 - w1 - w2
        if w3 < 0.05:
            continue
        blend = w1 * oof_xgb + w2 * oof_lgb + w3 * oof_cat
        s = balanced_accuracy_score(y_train, blend.argmax(axis=1))
        if s > best_w_score:
            best_w_score = s
            best_w = (w1, w2, w3)

w_xgb, w_lgb, w_cat = best_w
blend_opt = w_xgb * oof_xgb + w_lgb * oof_lgb + w_cat * oof_cat

print(f"\n균등 블렌딩 OOF:   {score_equal:.5f}")
print(f"최적 블렌딩 OOF:   {best_w_score:.5f} (XGB={w_xgb:.2f}, LGB={w_lgb:.2f}, CAT={w_cat:.2f})")

모델                     OOF
XGBoost            0.97542
LightGBM           0.97514
CatBoost           0.97738

균등 블렌딩 OOF:   0.97617
최적 블렌딩 OOF:   0.97727 (XGB=0.10, LGB=0.05, CAT=0.85)


In [8]:
# Bias Tuning (logit 공간에서 클래스별 편향 조정)
def get_tuned_preds(probs, bias):
    adjusted = logit(np.clip(probs, 1e-15, 1.0 - 1e-15)) + bias
    return np.argmax(adjusted, axis=1)

def tune_bias(probs, y_true):
    best_bias = np.zeros(3)
    best_score = balanced_accuracy_score(y_true, get_tuned_preds(probs, best_bias))
    print(f"  Raw score: {best_score:.5f}")

    for step in [1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002]:
        improved = True
        while improved:
            improved = False
            for ci in range(3):
                for d in [1, -1]:
                    trial_bias = best_bias.copy()
                    trial_bias[ci] += d * step
                    s = balanced_accuracy_score(y_true, get_tuned_preds(probs, trial_bias))
                    if s > best_score + 1e-9:
                        best_bias = trial_bias
                        best_score = s
                        improved = True
        print(f"  step={step:.3f}: score={best_score:.5f}, bias={best_bias.round(3)}")
    return best_bias, best_score

print("=== Bias Tuning (최적 블렌딩 기준) ===")
best_bias, tuned_score = tune_bias(blend_opt, y_train)
print(f"\n최종 튜닝된 OOF: {tuned_score:.5f} (튜닝 전 {best_w_score:.5f}, +{tuned_score-best_w_score:.5f})")

=== Bias Tuning (최적 블렌딩 기준) ===
  Raw score: 0.97727
  step=1.000: score=0.97878, bias=[ 2. -1. -1.]
  step=0.500: score=0.97880, bias=[ 2.  -0.5 -1. ]
  step=0.200: score=0.97882, bias=[ 1.8 -0.5 -1. ]
  step=0.100: score=0.97883, bias=[ 1.8 -0.6 -1. ]
  step=0.050: score=0.97884, bias=[ 1.75 -0.65 -1.  ]
  step=0.020: score=0.97885, bias=[ 1.77 -0.63 -1.  ]
  step=0.010: score=0.97886, bias=[ 1.76 -0.63 -1.  ]
  step=0.005: score=0.97886, bias=[ 1.76 -0.63 -1.  ]
  step=0.002: score=0.97886, bias=[ 1.76 -0.63 -1.  ]

최종 튜닝된 OOF: 0.97886 (튜닝 전 0.97727, +0.00160)


## 5. 제출 파일 생성

In [9]:
# 테스트 블렌딩
test_blend_opt = w_xgb * test_xgb + w_lgb * test_lgb + w_cat * test_cat

# Bias 적용한 테스트 예측
test_blend_tuned_labels = get_tuned_preds(test_blend_opt, best_bias)

submissions = {
    'exp006_xgb': test_xgb.argmax(axis=1),
    'exp006_blend': test_blend_opt.argmax(axis=1),
    'exp006_blend_biastuned': test_blend_tuned_labels,
}

for name, label_idx in submissions.items():
    labels = target_le.inverse_transform(label_idx)
    sub = pd.DataFrame({'id': test['id'], 'Irrigation_Need': labels})
    path = f'../submissions/submission_{name}.csv'
    sub.to_csv(path, index=False)
    dist = sub['Irrigation_Need'].value_counts()
    print(f"{path}")
    print(f"  {dict(dist)}\n")

print("=" * 55)
print("최종 결과 요약")
print("=" * 55)
print(f"XGBoost OOF:       {xgb_score:.5f}")
print(f"LightGBM OOF:      {lgb_score:.5f}")
print(f"CatBoost OOF:      {cat_score:.5f}")
print(f"균등 블렌딩 OOF:    {score_equal:.5f}")
print(f"최적 블렌딩 OOF:    {best_w_score:.5f}")
print(f"+Bias Tuning OOF:  {tuned_score:.5f}")

../submissions/submission_exp006_xgb.csv
  {'Low': np.int64(159577), 'Medium': np.int64(101304), 'High': np.int64(9119)}

../submissions/submission_exp006_blend.csv
  {'Low': np.int64(159481), 'Medium': np.int64(101141), 'High': np.int64(9378)}

../submissions/submission_exp006_blend_biastuned.csv
  {'Low': np.int64(159585), 'Medium': np.int64(99885), 'High': np.int64(10530)}

최종 결과 요약
XGBoost OOF:       0.97542
LightGBM OOF:      0.97514
CatBoost OOF:      0.97738
균등 블렌딩 OOF:    0.97617
최적 블렌딩 OOF:    0.97727
+Bias Tuning OOF:  0.97886
